EXTRACCION


In [ ]:
import pandas as pd
import sqlite3

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [45]:
df_categorias = pd.read_csv("/content/drive/MyDrive/LABS/00_DATASETS/categorias.csv")

df_productos = pd.read_csv(
    "/content/drive/MyDrive/LABS/00_DATASETS/productos.csv",
    encoding="latin1"
)

df_clientes = pd.read_excel("/content/drive/MyDrive/LABS/00_DATASETS/clientes.xlsx")

In [46]:
conexion = sqlite3.connect("/content/drive/MyDrive/LABS/00_DATASETS/northwind.db")

df_orders = pd.read_sql_query("SELECT * FROM Orders", conexion)
df_orders_det = pd.read_sql_query("SELECT * FROM 'Order Details'", conexion)
df_empleados = pd.read_sql_query("SELECT * FROM Employees", conexion)

TRANSFORMACION


In [50]:
def convertir_mayusculas(df):
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].str.upper()
    return df

In [48]:
df_categorias = convertir_mayusculas(df_categorias)
df_clientes = convertir_mayusculas(df_clientes)
df_productos = convertir_mayusculas(df_productos)
df_orders = convertir_mayusculas(df_orders)
df_orders_det = convertir_mayusculas(df_orders_det)

In [52]:
df_orders = df_orders[df_orders["ShippedDate"] != "NONE"]

In [53]:
df_orders["OrderDate"] = pd.to_datetime(df_orders["OrderDate"])
df_orders["ShippedDate"] = pd.to_datetime(df_orders["ShippedDate"])

In [54]:
df_empleados["nombre_completo"] = df_empleados["LastName"] + " " + df_empleados["FirstName"]
df_empleados["nombre_completo"] = df_empleados["nombre_completo"].str.upper()

In [55]:
df_orders["dias_prepara"] = (df_orders["ShippedDate"] - df_orders["OrderDate"]).dt.days

In [56]:
df_orders_det["subtotal"] = df_orders_det["Quantity"] * df_orders_det["UnitPrice"]

In [57]:
df_orders_det["totdescuento"] = df_orders_det["Quantity"] * df_orders_det["UnitPrice"] * df_orders_det["Discount"]

In [58]:
df_orders_det["total"] = df_orders_det["subtotal"] - df_orders_det["totdescuento"]

CARGA

In [59]:
df_prod_cat = pd.merge(df_productos, df_categorias, on="CategoryID")

df_temp = pd.merge(df_orders_det, df_prod_cat, on="ProductID")

df_temp = pd.merge(df_temp, df_orders, on="OrderID")

df_temp = pd.merge(df_temp, df_empleados, on="EmployeeID")

In [60]:
df_analitico = df_temp[[
    "OrderID",
    "ProductName",
    "CategoryName",
    "total",
    "subtotal",
    "totdescuento",
    "OrderDate",
    "dias_prepara",
    "UnitPrice",
    "Quantity",
    "nombre_completo"
]]

In [61]:
df_analitico.head(10)

,OrderID,ProductName,CategoryName,total,subtotal,totdescuento,OrderDate,dias_prepara,UnitPrice,Quantity,nombre_completo
0,10248,QUESO CABRALES,DAIRY PRODUCTS,168.00,168.0,0.00,2016-07-04,12.0,14.0,12,BUCHANAN STEVEN
1,10248,SINGAPOREAN HOKKIEN FRIED MEE,GRAINS/CEREALS,98.00,98.0,0.00,2016-07-04,12.0,9.8,10,BUCHANAN STEVEN
2,10248,MOZZARELLA DI GIOVANNI,DAIRY PRODUCTS,174.00,174.0,0.00,2016-07-04,12.0,34.8,5,BUCHANAN STEVEN
3,10249,TOFU,PRODUCE,167.40,167.4,0.00,2016-07-05,5.0,18.6,9,SUYAMA MICHAEL
4,10249,MANJIMUP DRIED APPLES,PRODUCE,1696.00,1696.0,0.00,2016-07-05,5.0,42.4,40,SUYAMA MICHAEL
5,10250,JACK'S NEW ENGLAND CLAM CHOWDER,SEAFOOD,77.00,77.0,0.00,2016-07-08,4.0,7.7,10,PEACOCK MARGARET
6,10250,MANJIMUP DRIED APPLES,PRODUCE,1261.40,1484.0,222.60,2016-07-08,4.0,42.4,35,PEACOCK MARGARET
7,10250,LOUISIANA FIERY HOT PEPPER SAUCE,CONDIMENTS,214.20,252.0,37.80,2016-07-08,4.0,16.8,15,PEACOCK MARGARET
8,10251,GUSTAF'S KNÄCKEBRÖD,GRAINS/CEREALS,95.76,100.8,5.04,2016-07-08,7.0,16.8,6,LEVERLING JANET
9,10251,RAVIOLI ANGELO,GRAINS/CEREALS,222.30,234.0,11.70,2016-07-08,7.0,15.6,15,LEVERLING JANET
